## 10. Extract evidence and perform safety reasoning

This block adds an interpretation layer on top of the model predictions. Up to this point, the notebook has trained GNN models that can score drug–disease pairs, but a high model score alone is not enough for a repurposing recommendation. To make the predictions more useful and explainable, this block extracts structured graph evidence that can support a drug candidate and also checks for safety-related warning signals.

### 1. Build node name lookups

The block starts by creating fast lookup dictionaries that map node indices back to readable names and raw IDs. These are built mainly from `nodes_master_df`, which was created earlier as the unified node table.

This is necessary because most of the graph logic in the notebook works with integer node indices for efficiency, but later evidence tables and explanations need human-readable labels such as disease names, protein names, and drug names.

The code also supplements the lookup from the edge table in case any nodes are missing from the first pass. By the end of this step, helper functions such as `get_node_name(...)` can convert graph indices into readable display names.

### 2. Detect important edge type groups

Since the graph is heterogeneous and contains many relation types, the block next identifies which edge types belong to which biological role. Instead of hard-coding everything in one place, it groups edge types into categories such as:

- drug–protein relations
- disease–protein relations
- protein–protein interaction edges
- disease–disease hierarchy or similarity edges
- disease–phenotype relations
- drug–effect or side-effect relations
- contraindication edges
- therapeutic indication edges

### 3. Build adjacency indexes for the graph

After identifying the edge groups, the block constructs adjacency dictionaries from the training graph. These indexes are built in both directions:

- **outgoing adjacency**
- **incoming adjacency**


### 4. Build therapeutic lookup tables from training data

The block then creates two lookup structures from the training indication edges:

- **`TRAIN_THERAPEUTIC_BY_DISEASE`**: maps each disease to the drugs already known to treat it
- **`TRAIN_DISEASES_BY_DRUG`**: maps each drug to the diseases it already treats

These lookups are used later for two main reasons:

- to identify what a drug is already known to treat
- to support disease-to-disease transfer evidence, where a drug may be considered plausible for a query disease because it already treats a biologically related disease

This gives the later reranking steps an explicit way to connect repurposing suggestions to existing therapeutic knowledge in the graph.

### 5. Define a model-scoring helper

Before extracting evidence, the block defines a helper function called `score_all_drugs_for_disease(...)`. This function uses the selected trained model, either the baseline model or the HGT model, to score all drug candidates for a chosen disease.

The result is a ranked dataframe of candidate drugs with their raw model scores. This is useful because the notebook does not want to explain every possible drug. Instead, it first uses the trained model to identify promising candidates and then attaches evidence and safety analysis to those candidates.

### 6. Extract supportive evidence paths

This is the core evidence-building section of the block. It defines the graph patterns that count as possible supporting evidence for a drug–disease recommendation.

#### Shared protein bridge

This captures cases where:
- a drug connects to a protein
- the disease is also associated with that same protein

This is one of the strongest and most direct evidence types because it suggests a shared molecular target or mechanism.

Conceptually, the pattern is:

**drug → protein ← disease**

#### PPI bridge

This captures indirect molecular support through the protein–protein interaction network. In this case:
- the drug connects to protein A
- protein A interacts with protein B
- the disease is associated with protein B

This provides a less direct but still biologically meaningful connection between the drug and disease.

Conceptually, the pattern is:

**drug → protein A — PPI — protein B ← disease**

#### Disease-to-disease transfer evidence

This looks for diseases related to the query disease through ontology or disease hierarchy edges and then checks whether the candidate drug already treats one of those related diseases.

This supports the idea of repurposing through disease similarity: if a drug already treats a related condition, it may also be relevant for the query disease.

Conceptually, the pattern is:

**drug → related disease ← disease hierarchy → query disease**

#### Effect or phenotype overlap

This checks whether:
- the drug connects to a phenotype or effect
- the disease also connects to the same phenotype

This is used as another form of evidence suggesting that the drug may interact with biology relevant to the disease presentation.

Conceptually, the pattern is:

**drug → phenotype/effect ← disease**

Together, these extractors build a broader evidence picture rather than relying on only one mechanistic route.

### 7. Extract safety signals

In addition to supportive evidence, the block also searches for graph patterns that may indicate risk.

#### Direct contraindication

This is the clearest safety warning. If the graph already contains a contraindication edge between the candidate drug and the disease, that is treated as a strong negative signal.

Conceptually:

**drug — contraindication → disease**

#### Adverse effect overlap

This safety pattern checks whether:
- the drug has a known adverse effect
- the disease is associated with the same phenotype or symptom

This can indicate that the drug may worsen or conflict with the disease state.

Conceptually:

**drug → adverse effect ← disease phenotype**

These safety signals are not necessarily used to reject a candidate automatically, but they become important penalties or flags in the later reranking blocks.

### 8. Build the main evidence bundle

After defining all evidence and safety extractors, the block combines them into a main wrapper function called `extract_evidence_bundle(...)`.

Given a disease and a candidate drug, this function collects:
- supportive evidence paths
- safety signals
- metadata such as node names and edge details
- counts and summaries that are easier to use later

This evidence bundle acts as a structured package containing the full reasoning context for a recommendation.

The block also defines helper functions to convert the bundle into more usable forms:

- **`evidence_bundle_to_tables(bundle)`** converts the bundle into clean evidence and safety dataframes
- **`bundle_to_rag_json(bundle)`** converts the bundle into a JSON-ready format that can later be passed into a reporting or explanation system

### 9. Run Demo

At the end of the block, a small demo section tests the evidence extraction pipeline on a sample disease query and top-ranked candidate drugs. This helps verify that the functions work correctly and gives an early preview of the kind of evidence that will later appear in the final recommendation tables and explanation outputs.

In [ ]:
# Evidence Extraction & Safety Reasoning

# Guards
assert "train_data"         in globals(), "Run Block 06 first."
assert "TARGET_EDGE_TRIPLET" in globals(), "Run Block 06 first."
assert "nodes_master_df"    in globals(), "Run Block 04 first."
assert "baseline_model"     in globals() or "hgt_model" in globals(), \
    "Run Block 07 (baseline) or Block 08 (HGT) first."

graph         = train_data
SUP_EDGE_TYPE = TARGET_EDGE_TRIPLET
SUP_SRC_TYPE, SUP_REL_TYPE, SUP_DST_TYPE = SUP_EDGE_TYPE

if "hgt_model" in globals():
    evidence_model      = hgt_model
    EVIDENCE_MODEL_NAME = "HGT"
else:
    evidence_model      = baseline_model
    EVIDENCE_MODEL_NAME = "Baseline"

print(f"Evidence model    : {EVIDENCE_MODEL_NAME}")
print(f"Supervision edge  : {SUP_EDGE_TYPE}")


# 1. Node name lookup

_NODE_NAME_LOOKUP: dict[tuple, str] = {}
_NODE_ID_LOOKUP:   dict[tuple, str] = {}

for _, row in nodes_master_df.iterrows():
    ntype = str(row["node_type"]).strip().lower()
    nidx  = int(row["node_index_within_type"])
    name  = str(row.get("node_name", "")).strip()
    raw_id = str(row.get("node_id_raw", "")).strip()
    if name:
        _NODE_NAME_LOOKUP[(ntype, nidx)] = name
    if raw_id:
        _NODE_ID_LOOKUP[(ntype, nidx)] = raw_id

# Supplement from edge table if any nodes were missed
for col_prefix in [("src", "src_index"), ("dst", "dst_index")]:
    prefix, idx_col = col_prefix
    for _, row in edges_model_ready_df.iterrows():
        ntype = str(row[f"{prefix}_type"]).strip().lower()
        nidx  = int(row[idx_col])
        name  = str(row.get(f"{prefix}_name", "")).strip()
        if name and (ntype, nidx) not in _NODE_NAME_LOOKUP:
            _NODE_NAME_LOOKUP[(ntype, nidx)] = name

def get_node_name(node_type: str, node_idx: int) -> str:
    """
    Return 'DisplayName (type_idx)' for a node.
    Raises KeyError if no name is found — this should never happen
    if nodes_master_df covers all nodes.
    """
    key = (str(node_type).strip().lower(), int(node_idx))
    name = _NODE_NAME_LOOKUP.get(key)
    if not name:
        raise KeyError(f"No name found for {key}. Check nodes_master_df coverage.")
    return f"{name} ({node_type}_{node_idx})"

def get_node_id(node_type: str, node_idx: int):
    """Return the raw database ID (e.g. DrugBank ID) for a node."""
    key = (str(node_type).strip().lower(), int(node_idx))
    return _NODE_ID_LOOKUP.get(key)

# Coverage check — drugs and diseases must be fully covered
_coverage = []
for ntype in graph.node_types:
    n       = int(graph[ntype].num_nodes)
    found   = sum((ntype.lower(), i) in _NODE_NAME_LOOKUP for i in range(n))
    _coverage.append({"node_type": ntype, "num_nodes": n,
                       "names_found": found,
                       "coverage_pct": round(100.0 * found / max(n, 1), 2)})

node_name_coverage_df = pd.DataFrame(_coverage)
print("\nNode-name coverage:")
display(node_name_coverage_df)

for core_t in [SUP_SRC_TYPE, SUP_DST_TYPE]:
    row = node_name_coverage_df[node_name_coverage_df["node_type"] == core_t]
    assert len(row) > 0 and int(row["names_found"].iloc[0]) > 0, \
        f"No names found for required type: {core_t}"


# 2. Edge type group detection

def _norm(s: str) -> str:
    return str(s).lower().replace("-", "_").replace(" ", "_")

def _type_has(node_type: str, keys: list) -> bool:
    t = _norm(node_type)
    return any(k in t for k in keys)

def _rel_has(relation: str, keys: list) -> bool:
    r = _norm(relation)
    return any(k in r for k in keys)

def _find_edge_types(src_keys, dst_keys, rel_keys=None, allow_reverse=False):
    """Return edge types matching src_keys on source, dst_keys on dest."""
    found = []
    for et in graph.edge_types:
        s, r, d = et
        s_ok = _type_has(s, src_keys)
        d_ok = _type_has(d, dst_keys)
        r_ok = True if rel_keys is None else _rel_has(r, rel_keys)
        if s_ok and d_ok and r_ok:
            found.append(et)
        elif allow_reverse and _type_has(s, dst_keys) and _type_has(d, src_keys) and r_ok:
            found.append(et)
    return found

_DRUG_K    = ["drug", "compound"]
_DIS_K     = ["disease", "disorder"]
_PROT_K    = ["protein", "gene"]
_PHENO_K   = ["phenotype", "effect", "symptom", "adverse"]

drug_protein_edge_types    = _find_edge_types(_DRUG_K, _PROT_K, None, allow_reverse=True)
disease_protein_edge_types = _find_edge_types(_DIS_K,  _PROT_K, None, allow_reverse=True)
ppi_edge_types             = _find_edge_types(_PROT_K, _PROT_K, ["ppi", "interact"])
disease_disease_edge_types = _find_edge_types(_DIS_K,  _DIS_K,  None)
drug_effect_edge_types     = _find_edge_types(_DRUG_K, _PHENO_K, None, allow_reverse=True)
disease_effect_edge_types  = _find_edge_types(_DIS_K,  _PHENO_K, None, allow_reverse=True)
contra_edge_types          = _find_edge_types(_DRUG_K, _DIS_K, ["contra"], allow_reverse=True)

# The supervision edge is always a therapeutic edge
therapeutic_edge_types = list({SUP_EDGE_TYPE})

print("\nDetected edge type groups:")
for name, ets in [
    ("drug_protein",    drug_protein_edge_types),
    ("disease_protein", disease_protein_edge_types),
    ("ppi",             ppi_edge_types),
    ("disease_disease", disease_disease_edge_types),
    ("drug_effect",     drug_effect_edge_types),
    ("disease_effect",  disease_effect_edge_types),
    ("contraindication",contra_edge_types),
    ("therapeutic",     therapeutic_edge_types),
]:
    print(f"  {name:<18}: {len(ets)} edge types")


# 3. Adjacency indexes

EDGE_INDEX_OUT: dict = {}   # {edge_type: {src_idx: [dst_idx, ...]}}
EDGE_INDEX_IN:  dict = {}   # {edge_type: {dst_idx: [src_idx, ...]}}

for et in graph.edge_types:
    ei      = graph[et].edge_index.detach().cpu()
    out_map = defaultdict(list)
    in_map  = defaultdict(list)
    for s, d in ei.t().tolist():
        out_map[int(s)].append(int(d))
        in_map[int(d)].append(int(s))
    EDGE_INDEX_OUT[et] = out_map
    EDGE_INDEX_IN[et]  = in_map

def get_neighbors(node_type: str, node_idx: int,
                  edge_types: list = None) -> list:
    """
    Return list of {neighbor_type, neighbor_idx, relation, direction}
    for all edges incident to (node_type, node_idx).
    """
    rows = []
    node_idx = int(node_idx)
    for et in (edge_types or graph.edge_types):
        s_type, rel, d_type = et
        if node_type == s_type:
            for nbr in EDGE_INDEX_OUT[et].get(node_idx, []):
                rows.append({"neighbor_type": d_type, "neighbor_idx": int(nbr),
                              "relation": rel, "edge_type": et, "direction": "out"})
        if node_type == d_type:
            for nbr in EDGE_INDEX_IN[et].get(node_idx, []):
                rows.append({"neighbor_type": s_type, "neighbor_idx": int(nbr),
                              "relation": rel, "edge_type": et, "direction": "in"})
    return rows

def shared_intermediates(left_type, left_idx, right_type, right_idx,
                         left_ets, right_ets, allowed_type_keys=None, top_n=5):
    """
    Find nodes that are neighbors of both (left_type, left_idx) and
    (right_type, right_idx) via their respective edge type sets.
    Returns up to top_n shared intermediate node records.
    """
    left_nbrs  = get_neighbors(left_type,  left_idx,  left_ets)
    right_nbrs = get_neighbors(right_type, right_idx, right_ets)

    if allowed_type_keys:
        left_nbrs  = [r for r in left_nbrs  if _type_has(r["neighbor_type"], allowed_type_keys)]
        right_nbrs = [r for r in right_nbrs if _type_has(r["neighbor_type"], allowed_type_keys)]

    left_map  = defaultdict(list)
    right_map = defaultdict(list)
    for r in left_nbrs:
        left_map[(r["neighbor_type"], int(r["neighbor_idx"]))].append(r)
    for r in right_nbrs:
        right_map[(r["neighbor_type"], int(r["neighbor_idx"]))].append(r)

    shared = list(set(left_map) & set(right_map))[:top_n]
    out = []
    for key in shared:
        interm_type, interm_idx = key
        l = left_map[key][0]
        r = right_map[key][0]
        out.append({
            "intermediate_type": interm_type,
            "intermediate_idx":  int(interm_idx),
            "intermediate_name": get_node_name(interm_type, interm_idx),
            "left_relation":     l["relation"],
            "right_relation":    r["relation"],
        })
    return out


# 4. Training therapeutic lookups

TRAIN_THERAPEUTIC_BY_DISEASE: dict[int, set] = defaultdict(set)
TRAIN_DISEASES_BY_DRUG:       dict[int, set] = defaultdict(set)

ei_train = graph[SUP_EDGE_TYPE].edge_index.detach().cpu()
for drug_idx, dis_idx in ei_train.t().tolist():
    TRAIN_THERAPEUTIC_BY_DISEASE[int(dis_idx)].add(int(drug_idx))
    TRAIN_DISEASES_BY_DRUG[int(drug_idx)].add(int(dis_idx))

print(f"\nTraining therapeutic pairs : {sum(len(v) for v in TRAIN_THERAPEUTIC_BY_DISEASE.values()):,}")


# 5. Model scoring helper

@torch.no_grad()
def score_all_drugs_for_disease(model, graph, disease_idx: int,
                                edge_type: tuple = None) -> pd.DataFrame:
    """
    Score every drug against one disease using the model.
    Returns DataFrame[drug_idx, score] sorted descending.
    """
    if edge_type is None:
        edge_type = model.supervision_edge_type
    s_type, _, d_type = edge_type
    dev  = next(model.parameters()).device
    n    = int(graph[s_type].num_nodes)
    ei   = torch.stack([
        torch.arange(n, device=dev),
        torch.full((n,), disease_idx, dtype=torch.long, device=dev)
    ])
    model.eval()
    z    = model.encode(graph.to(dev))
    logits = model.decode(z, ei, edge_type=edge_type)
    scores = torch.sigmoid(logits).cpu().numpy()
    return (pd.DataFrame({"drug_idx": np.arange(n), "score": scores})
              .sort_values("score", ascending=False)
              .reset_index(drop=True))

@torch.no_grad()
def score_pair(model, graph, drug_idx: int, disease_idx: int) -> float:
    """Score a single (drug, disease) pair."""
    dev = next(model.parameters()).device
    ei  = torch.tensor([[drug_idx], [disease_idx]], dtype=torch.long, device=dev)
    model.eval()
    z   = model.encode(graph.to(dev))
    return float(torch.sigmoid(model.decode(z, ei)).item())


# 6. Evidence extractors

def _fmt_path(*parts) -> str:
    """Format a path as 'A --[rel]--> B ...' string."""
    return " ".join(str(p) for p in parts)

def extract_shared_protein_evidence(drug_idx: int, disease_idx: int,
                                    top_n: int = 5) -> list:
    """
    Find proteins targeted by the drug that are also associated with
    the disease.  This is the strongest mechanistic evidence type —
    it shows the drug acts on a biological target directly relevant
    to the disease.
    """
    rows = shared_intermediates(
        SUP_SRC_TYPE, drug_idx,
        SUP_DST_TYPE, disease_idx,
        drug_protein_edge_types, disease_protein_edge_types,
        allowed_type_keys=_PROT_K, top_n=top_n,
    )
    out = []
    for r in rows:
        out.append({
            "evidence_type":      "shared_protein_bridge",
            "drug_idx":           int(drug_idx),
            "disease_idx":        int(disease_idx),
            "intermediate_type":  r["intermediate_type"],
            "intermediate_idx":   r["intermediate_idx"],
            "intermediate_name":  r["intermediate_name"],
            "path_text": _fmt_path(
                get_node_name(SUP_SRC_TYPE, drug_idx),
                f"--[{r['left_relation']}]-->",
                r["intermediate_name"],
                f"<--[{r['right_relation']}]--",
                get_node_name(SUP_DST_TYPE, disease_idx),
            ),
            "score_hint": 3.0,
        })
    return out


def extract_ppi_bridge_evidence(drug_idx: int, disease_idx: int,
                                top_n: int = 5) -> list:
    """
    Find 2-hop protein paths: drug → protein A — PPI — protein B ← disease.
    The PPI hop extends the drug's reach to proteins it doesn't directly
    target but is functionally connected to through the interaction network.
    """
    drug_prots    = [r for r in get_neighbors(SUP_SRC_TYPE, drug_idx, drug_protein_edge_types)
                     if _type_has(r["neighbor_type"], _PROT_K)]
    disease_prots = {(r["neighbor_type"], int(r["neighbor_idx"])): r["relation"]
                     for r in get_neighbors(SUP_DST_TYPE, disease_idx, disease_protein_edge_types)
                     if _type_has(r["neighbor_type"], _PROT_K)}

    out  = []
    seen = set()

    for dp in drug_prots:
        p1_type, p1_idx = dp["neighbor_type"], int(dp["neighbor_idx"])
        for et in ppi_edge_types:
            s_t, rel, d_t = et
            if p1_type == s_t:
                nbrs     = EDGE_INDEX_OUT[et].get(p1_idx, [])
                nbr_type = d_t
            elif p1_type == d_t:
                nbrs     = EDGE_INDEX_IN[et].get(p1_idx, [])
                nbr_type = s_t
            else:
                continue

            for p2_idx in nbrs:
                key = (nbr_type, int(p2_idx))
                if key not in disease_prots or (p1_idx, p2_idx) in seen:
                    continue
                seen.add((p1_idx, p2_idx))
                out.append({
                    "evidence_type":      "ppi_bridge",
                    "drug_idx":           int(drug_idx),
                    "disease_idx":        int(disease_idx),
                    "intermediate_type":  nbr_type,
                    "intermediate_idx":   int(p2_idx),
                    "intermediate_name":  get_node_name(nbr_type, p2_idx),
                    "path_text": _fmt_path(
                        get_node_name(SUP_SRC_TYPE, drug_idx),
                        f"--[{dp['relation']}]-->",
                        get_node_name(p1_type, p1_idx),
                        f"--[{rel}]-->",
                        get_node_name(nbr_type, p2_idx),
                        f"<--[{disease_prots[key]}]--",
                        get_node_name(SUP_DST_TYPE, disease_idx),
                    ),
                    "score_hint": 2.5,
                })
                if len(out) >= top_n:
                    return out
    return out


def extract_disease_transfer_evidence(drug_idx: int, disease_idx: int,
                                      top_n: int = 5) -> list:
    """
    If the drug treats a related disease D', and D' is connected to the
    query disease in the disease hierarchy, infer a repurposing rationale.
    This is 'guilt by association' at the disease level.
    """
    out  = []
    seen = set()

    for other_dis in sorted(TRAIN_DISEASES_BY_DRUG.get(int(drug_idx), set())):
        for et in disease_disease_edge_types:
            s_t, rel, d_t = et
            if not (_type_has(s_t, _DIS_K) and _type_has(d_t, _DIS_K)):
                continue
            connected = (
                disease_idx in EDGE_INDEX_OUT[et].get(other_dis, []) or
                disease_idx in EDGE_INDEX_IN[et].get(other_dis, [])
            )
            if connected and (other_dis, rel) not in seen:
                seen.add((other_dis, rel))
                out.append({
                    "evidence_type":       "disease_disease_transfer",
                    "drug_idx":            int(drug_idx),
                    "disease_idx":         int(disease_idx),
                    "similar_disease_idx": int(other_dis),
                    "similar_disease_name": get_node_name(SUP_DST_TYPE, other_dis),
                    "path_text": _fmt_path(
                        get_node_name(SUP_SRC_TYPE, drug_idx),
                        "--[known treatment]-->",
                        get_node_name(SUP_DST_TYPE, other_dis),
                        f"--[{rel}]-->",
                        get_node_name(SUP_DST_TYPE, disease_idx),
                    ),
                    "score_hint": 2.2,
                })
                if len(out) >= top_n:
                    return out
    return out


def extract_effect_overlap_evidence(drug_idx: int, disease_idx: int,
                                    top_n: int = 5) -> list:
    """
    Find phenotypes / effects shared between the drug's known side effects
    and the disease's known phenotypes.  A drug that produces some of the
    same phenotypic effects as the disease does may be modulating a relevant
    pathway.
    """
    rows = shared_intermediates(
        SUP_SRC_TYPE, drug_idx,
        SUP_DST_TYPE, disease_idx,
        drug_effect_edge_types, disease_effect_edge_types,
        allowed_type_keys=_PHENO_K, top_n=top_n,
    )
    return [{
        "evidence_type":     "effect_or_phenotype_overlap",
        "drug_idx":          int(drug_idx),
        "disease_idx":       int(disease_idx),
        "intermediate_type": r["intermediate_type"],
        "intermediate_idx":  r["intermediate_idx"],
        "intermediate_name": r["intermediate_name"],
        "path_text": _fmt_path(
            get_node_name(SUP_SRC_TYPE, drug_idx),
            f"--[{r['left_relation']}]-->",
            r["intermediate_name"],
            f"<--[{r['right_relation']}]--",
            get_node_name(SUP_DST_TYPE, disease_idx),
        ),
        "score_hint": 1.5,
    } for r in rows]


# 7. Safety extractors

def extract_contraindications(drug_idx: int, disease_idx: int) -> list:
    """
    Check whether the graph contains a direct contraindication edge
    between this drug and this disease.  A high-severity safety signal.
    """
    out = []
    for et in contra_edge_types:
        s_t, rel, d_t = et
        hit = False
        if s_t == SUP_SRC_TYPE and d_t == SUP_DST_TYPE:
            hit = disease_idx in EDGE_INDEX_OUT[et].get(drug_idx, [])
        elif d_t == SUP_SRC_TYPE and s_t == SUP_DST_TYPE:
            hit = disease_idx in EDGE_INDEX_IN[et].get(drug_idx, [])
        if hit:
            out.append({
                "safety_type": "direct_contraindication",
                "severity":    "high",
                "relation":    rel,
                "text": (
                    f"Direct contraindication: "
                    f"{get_node_name(SUP_SRC_TYPE, drug_idx)} "
                    f"--[{rel}]-- "
                    f"{get_node_name(SUP_DST_TYPE, disease_idx)}"
                ),
            })
    return out


def extract_adverse_overlap(drug_idx: int, disease_idx: int,
                            top_n: int = 5) -> list:
    """
    Flag phenotypes that are both drug side effects and disease phenotypes.
    Giving a drug whose side effects mimic the disease's symptoms could
    worsen the patient's condition.
    """
    rows = shared_intermediates(
        SUP_SRC_TYPE, drug_idx,
        SUP_DST_TYPE, disease_idx,
        drug_effect_edge_types, disease_effect_edge_types,
        allowed_type_keys=_PHENO_K, top_n=top_n,
    )
    return [{
        "safety_type": "adverse_effect_overlap",
        "severity":    "medium",
        "relation":    f"{r['left_relation']} / {r['right_relation']}",
        "text": (
            f"Phenotype/effect overlap: {r['intermediate_name']} — "
            f"drug via [{r['left_relation']}], disease via [{r['right_relation']}]"
        ),
    } for r in rows]


def _risk_level(safety_rows: list) -> str:
    """Summarise overall risk from a list of safety records."""
    if not safety_rows:
        return "low"
    return "high" if any(r["severity"] == "high" for r in safety_rows) else "medium"


# 8. Main evidence bundle

def extract_evidence_bundle(drug_idx: int, disease_idx: int,
                             model=None, top_n_each: int = 5) -> dict:
    """
    Run all four evidence extractors + two safety extractors for a single
    (drug, disease) pair.

    Returns a structured dict with:
      drug/disease metadata, model score, evidence rows, safety rows,
      evidence count summary, and a risk level string.
    """
    if model is None:
        model = evidence_model

    drug_idx    = int(drug_idx)
    disease_idx = int(disease_idx)
    pred_score  = score_pair(model, graph, drug_idx, disease_idx)

    # Collect and rank evidence by score_hint
    evidence_rows = sorted(
        extract_shared_protein_evidence(drug_idx, disease_idx, top_n_each) +
        extract_ppi_bridge_evidence(drug_idx, disease_idx, top_n_each) +
        extract_disease_transfer_evidence(drug_idx, disease_idx, top_n_each) +
        extract_effect_overlap_evidence(drug_idx, disease_idx, top_n_each),
        key=lambda r: r.get("score_hint", 0.0), reverse=True,
    )
    safety_rows = (
        extract_contraindications(drug_idx, disease_idx) +
        extract_adverse_overlap(drug_idx, disease_idx, top_n_each)
    )

    return {
        "drug_idx":      drug_idx,
        "drug_name":     get_node_name(SUP_SRC_TYPE, drug_idx),
        "drug_id":       get_node_id(SUP_SRC_TYPE, drug_idx),
        "drug_type":     SUP_SRC_TYPE,
        "disease_idx":   disease_idx,
        "disease_name":  get_node_name(SUP_DST_TYPE, disease_idx),
        "disease_id":    get_node_id(SUP_DST_TYPE, disease_idx),
        "disease_type":  SUP_DST_TYPE,
        "model_name":    EVIDENCE_MODEL_NAME,
        "pred_score":    float(pred_score),
        "evidence_counts": dict(Counter(r["evidence_type"] for r in evidence_rows)),
        "n_evidence_rows": len(evidence_rows),
        "n_safety_flags":  len(safety_rows),
        "risk_level":      _risk_level(safety_rows),
        "evidence_rows":   evidence_rows,
        "safety_rows":     safety_rows,
    }


def evidence_bundle_to_tables(bundle: dict):
    """Split a bundle into (evidence_df, safety_df) DataFrames."""
    ev_df = pd.DataFrame(bundle["evidence_rows"]) if bundle["evidence_rows"] else \
            pd.DataFrame(columns=["evidence_type", "intermediate_name",
                                  "similar_disease_name", "path_text", "score_hint"])
    sf_df = pd.DataFrame(bundle["safety_rows"]) if bundle["safety_rows"] else \
            pd.DataFrame(columns=["safety_type", "severity", "relation", "text"])
    return ev_df, sf_df


def bundle_to_rag_json(bundle: dict) -> dict:
    """Return a JSON-serialisable version of a bundle for LLM prompting."""
    return {
        "drug":    {"idx": bundle["drug_idx"],    "name": bundle["drug_name"],
                    "id":  bundle["drug_id"],     "type": bundle["drug_type"]},
        "disease": {"idx": bundle["disease_idx"], "name": bundle["disease_name"],
                    "id":  bundle["disease_id"],  "type": bundle["disease_type"]},
        "model_name":      bundle["model_name"],
        "pred_score":      bundle["pred_score"],
        "risk_level":      bundle["risk_level"],
        "evidence_counts": bundle["evidence_counts"],
        "evidence":        bundle["evidence_rows"],
        "safety":          bundle["safety_rows"],
    }


# 9. Quick demo

# Pick the first test-positive disease as demo target
_demo_ei  = globals().get("test_pos_edge_label_index")
_demo_dis = int(_demo_ei[1, 0].item()) if _demo_ei is not None else 0

print(f"\nDemo disease index : {_demo_dis}")
print(f"Demo disease name  : {get_node_name(SUP_DST_TYPE, _demo_dis)}")

# Suppress known train drugs so recommendations are novel
_demo_raw  = score_all_drugs_for_disease(evidence_model, graph, _demo_dis)
_train_known = TRAIN_THERAPEUTIC_BY_DISEASE.get(_demo_dis, set())
_demo_raw  = _demo_raw[~_demo_raw["drug_idx"].isin(_train_known)].head(5)

_demo_bundle = extract_evidence_bundle(
    int(_demo_raw.iloc[0]["drug_idx"]), _demo_dis,
    model=evidence_model, top_n_each=5,
)
_ev_df, _sf_df = evidence_bundle_to_tables(_demo_bundle)

print(f"\nTop candidate    : {_demo_bundle['drug_name']}")
print(f"Predicted score  : {_demo_bundle['pred_score']:.4f}")
print(f"Risk level       : {_demo_bundle['risk_level']}")
print(f"Evidence counts  : {_demo_bundle['evidence_counts']}")

print("\nEvidence table (first 5 rows):")
display(_ev_df.head())
print("\nSafety table:")
display(_sf_df)


# Summary
print(f"\n{'='*55}")
print(f"  BLOCK 10 COMPLETE")
print(f"{'='*55}")
print("  Shared functions available:")
print("    get_node_name(node_type, idx)")
print("    score_all_drugs_for_disease(model, graph, disease_idx)")
print("    extract_evidence_bundle(drug_idx, disease_idx, model)")
print("    evidence_bundle_to_tables(bundle)")
print("    bundle_to_rag_json(bundle)")
print("    TRAIN_THERAPEUTIC_BY_DISEASE / TRAIN_DISEASES_BY_DRUG")
print("    EDGE_INDEX_OUT / EDGE_INDEX_IN")
print("    drug/disease/ppi/effect/contra edge type group lists")
print(f"{'='*55}")

Evidence model    : HGT
Supervision edge  : ('drug', 'indication', 'disease')

Node-name coverage:


,node_type,num_nodes,names_found,coverage_pct
0,disease,17080,17080,100.0
1,drug,6681,6681,100.0
2,effect,990,990,100.0
3,phenotype,15311,15311,100.0
4,protein,19059,19059,100.0



Detected edge type groups:
  drug_protein      : 4 edge types
  disease_protein   : 1 edge types
  ppi               : 1 edge types
  disease_disease   : 1 edge types
  drug_effect       : 1 edge types
  disease_effect    : 1 edge types
  contraindication  : 1 edge types
  therapeutic       : 1 edge types

Training therapeutic pairs : 6,571

Demo disease index : 1479
Demo disease name  : psoriatic arthritis (disease_1479)

Top candidate    : Capecitabine (drug_1067)
Predicted score  : 0.5734
Risk level       : low
Evidence counts  : {}

Evidence table (first 5 rows):


,evidence_type,intermediate_name,similar_disease_name,path_text,score_hint



Safety table:


,safety_type,severity,relation,text



  BLOCK 10 COMPLETE
  Shared functions available:
    get_node_name(node_type, idx)
    score_all_drugs_for_disease(model, graph, disease_idx)
    extract_evidence_bundle(drug_idx, disease_idx, model)
    evidence_bundle_to_tables(bundle)
    bundle_to_rag_json(bundle)
    TRAIN_THERAPEUTIC_BY_DISEASE / TRAIN_DISEASES_BY_DRUG
    EDGE_INDEX_OUT / EDGE_INDEX_IN
    drug/disease/ppi/effect/contra edge type group lists


## 11. Evidence-aware drug reranking

This block takes the raw model predictions for a queried disease and turns them into a more interpretable recommendation list. Instead of ranking drugs only by the model’s predicted score, this section adds two additional considerations:

- **supporting graph evidence**, which rewards candidates that have stronger biological justification
- **safety signals**, which penalize candidates that show warning signs in the graph

The final reranking formula used in this block is:

$$
\boxed{
\begin{aligned}
\mathbf{\text{final_score}} =\, &W_{\text{MODEL}} \times \text{raw_model_score} \\
+\, &W_{\text{EVIDENCE}} \times \text{evidence_strength} \\
-\, &W_{\text{SAFETY}} \times \text{safety_penalty}
\end{aligned}
}
$$

This makes the recommendations more useful than a plain model ranking, because the top drugs are no longer chosen only by predictive confidence. They are also filtered through a basic layer of explanation and risk awareness.

### 1. User settings and setup

The block begins by checking that the required functions and lookup tables from earlier sections already exist, especially the evidence extraction pipeline from Block 10 and the training-treatment lookups derived from the graph.

It then defines the main user-controlled settings, including:

- `DISEASE_QUERY` — the disease name entered as free text
- `MODEL_CHOICE` — whether to use the **HGT** model or the **baseline**
- `RAW_POOL` — how many raw candidate drugs to consider before reranking
- `FINAL_TOP_K` — how many final recommendations to keep
- `EVIDENCE_TOP_N` — how many evidence items to retain per evidence family
- `EXCLUDE_TRAIN` — whether to remove drugs already known to treat the disease in training
- `MIN_SCORE` — an optional minimum raw model score

The block also defines the weight values used in the reranking formula, along with finer-grained weights for specific evidence families and safety penalties.


### 2. Select the model to use

The next step resolves which trained model should drive the ranking.

If `MODEL_CHOICE` is set to `"hgt"`, the block uses the HGT model trained in Block 8.  
If it is set to `"baseline"`, it uses the baseline heterogeneous GNN from Block 7.

The selected model is stored in:
- `selected_model`
- `selected_model_name`

This is useful because the reranking logic itself stays the same regardless of which predictive model is used underneath.

### 3. Match the disease query to a graph disease node

Since the query is entered as free text, the notebook needs to map it to the correct disease node in the graph. This is handled by two helper functions:

- **`_clean(text)`** normalizes text by lowercasing it and removing punctuation
- **`match_disease(query)`** performs fuzzy matching against the disease nodes in `nodes_master_df`

The disease matcher scores candidates using multiple signals:
- exact match
- substring match
- edit-distance similarity
- token overlap

This makes the search more robust to formatting differences and slightly different naming conventions.

The top match is then chosen as the active disease for the rest of the block and stored in:
- `selected_disease_idx`
- `selected_disease_name`

### 4. Resolve known treatments for each drug

Before ranking candidates, the block defines a helper called `get_known_treatments(drug_idx)`. This function looks up the diseases that a drug already treats in the training graph and returns them as a readable string.

This is not part of the reranking score itself, but it is useful for interpretation. When a drug is recommended for a disease, it helps to know what that drug is already known to treat. For example, if the recommended drug already treats a related cardiovascular disease, that provides an intuitive repurposing rationale.

If the drug has no known training indications, the function labels it as:

- **`novel (no known indication)`**

That makes it easier to distinguish between:
- drugs being suggested through transfer from known related uses
- drugs that look like more novel repurposing candidates

### 5. Retrieve the raw candidate pool

Once the disease is matched and the model is selected, the block retrieves the raw model predictions using:

- `score_all_drugs_for_disease(selected_model, graph, selected_disease_idx)`

This produces a ranked list of candidate drugs and their raw model scores for the selected disease.

The block then optionally:
- removes drugs already known to treat the disease in training, if `EXCLUDE_TRAIN=True`
- filters by `MIN_SCORE`, if a minimum score threshold is provided

After that, it keeps only the top `RAW_POOL` candidates. This smaller pool is what the reranking step will analyze in more detail.

### 6. Compute evidence strength and safety penalty

This is the core scoring logic of the block.

Two helper functions are defined here:

#### `_evidence_strength(bundle)`

This function converts the evidence bundle from Block 10 into a weighted evidence score.

It looks at the number of evidence items in each evidence family, such as:
- shared protein bridges
- PPI bridges
- disease-to-disease transfer evidence
- effect or phenotype overlap

These evidence families are weighted differently. Shared protein bridges receive the highest weight because they are treated as the strongest mechanistic support, while phenotype overlap receives the lowest weight because it is more indirect.

The function returns:
- a raw evidence score
- a normalized evidence strength between 0 and 1
- the evidence counts for each family

#### `_safety_penalty(bundle)`

This function converts the safety rows in the evidence bundle into a penalty score.

It checks for:
- **direct contraindications**
- **adverse effect overlap**
- **zero-evidence cases**, where the candidate has no supporting evidence at all

These safety types are also weighted differently. Direct contraindications receive the strongest penalty, while lack of evidence is penalized more lightly.

The function returns:
- a raw safety score
- a normalized safety penalty between 0 and 1
- counts for the different safety signals

### 7. Rerank the candidate drugs

The main reranking loop iterates over every drug in the raw candidate pool.

For each candidate drug, the block:
1. reads the raw model score
2. calls `extract_evidence_bundle(...)` to build its supporting evidence and safety information
3. computes the normalized evidence strength
4. computes the normalized safety penalty
5. combines everything using the reranking formula

The resulting candidate record includes:
- drug index and name
- raw model score
- rerank score
- risk level
- number of evidence rows
- number of safety flags
- detailed evidence and safety counts

All of these candidate rows are collected into `reranked_df`.

The dataframe is then sorted primarily by:
- rerank score
- evidence strength
- raw model score

This makes the final ordering more biologically informed than the original model ranking alone.

### 8. Build the final top-K recommendation table

After the full candidate pool is reranked, the block takes the top `FINAL_TOP_K` drugs and stores them in:

- `final_topk_df`

It then adds a new column:

- `known_treatment_for`

using the `get_known_treatments(...)` helper described earlier.

This final top-K table is meant to be easy to inspect and present. It shows not only the reranked score, but also:
- the original model score
- the risk level
- how much evidence was found
- how many safety flags were triggered
- what each drug is already known to treat

### 9. Build detailed result objects for downstream use

In addition to the summary table, the block also constructs a richer per-drug result object for each top recommendation.

These are stored in:

- `final_detailed_results`

Each entry includes:
- the drug rank
- drug metadata
- model score
- rerank score
- risk level
- the full evidence bundle
- evidence and safety dataframes
- a JSON-formatted explanation bundle through `bundle_to_rag_json(...)`

This structured output is important because later blocks reuse it for:
- disease-aware reranking
- the final demo pipeline
- evidence display and explanation formatting

### 10. Display and summary outputs

At the end of the block, the notebook prints a formatted summary table for the selected disease and selected model. This table displays the top reranked candidates with columns such as:

- Drug
- Model Score
- Rerank Score
- Risk
- Evidence
- Safety Flags
- Known Treatment For

In [ ]:
# Evidence-Aware Reranking

# Guards
assert "extract_evidence_bundle"      in globals(), "Run Block 10 first."
assert "score_all_drugs_for_disease"  in globals(), "Run Block 10 first."
assert "TRAIN_THERAPEUTIC_BY_DISEASE" in globals(), "Run Block 10 first."
assert "TRAIN_DISEASES_BY_DRUG"       in globals(), "Run Block 10 first."
assert "nodes_master_df"              in globals(), "Run Block 04 first."


# User settings

DISEASE_QUERY  = "hypertension"   # free-text disease name
MODEL_CHOICE   = "hgt"            # "hgt" or "baseline"

RAW_POOL       = 100    # raw candidates to score before reranking
FINAL_TOP_K    = 5      # drugs to return after reranking
EVIDENCE_TOP_N = 5      # evidence items per family per drug
EXCLUDE_TRAIN  = True   # exclude drugs already known in training
MIN_SCORE      = None   # optional raw score floor (e.g. 0.4)

# Reranking weights
W_MODEL    = 0.65
W_EVIDENCE = 0.30
W_SAFETY   = 0.20

# Evidence family weights
W_SHARED_PROT = 3.0
W_PPI         = 2.5
W_DD_TRANSFER = 2.0
W_EFFECT      = 1.0

# Safety penalty weights
P_CONTRA   = 4.0
P_ADVERSE  = 1.5
P_ZERO_EV  = 1.5

MAX_KNOWN_DISEASES = 3


# 1. Model selection

_choice = MODEL_CHOICE.strip().lower()
if _choice == "hgt":
    assert "hgt_model" in globals(), "hgt_model not found — run Block 08 first."
    selected_model      = hgt_model
    selected_model_name = "HGT"
elif _choice == "baseline":
    assert "baseline_model" in globals(), "baseline_model not found — run Block 07 first."
    selected_model      = baseline_model
    selected_model_name = "Baseline"
else:
    raise ValueError("MODEL_CHOICE must be 'hgt' or 'baseline'.")

print(f"Using model : {selected_model_name}")


# 2. Disease name matching

def _clean(text: str) -> str:
    """Normalise text for matching: lowercase, strip punctuation."""
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9\s\-_/]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def match_disease(query: str, top_n: int = 10) -> pd.DataFrame:
    """
    Fuzzy-match a free-text query against all disease nodes in the graph.
    Returns a DataFrame of candidates sorted by weighted match score using
    four signals: exact match, substring match, edit-distance ratio, and
    token Jaccard overlap.
    """
    q_norm   = _clean(query)
    q_tokens = set(q_norm.split())

    catalog = nodes_master_df[
        (nodes_master_df["node_type"].str.lower() == SUP_DST_TYPE.lower()) &
        (nodes_master_df["node_index_within_type"].astype(int) <
         int(graph[SUP_DST_TYPE].num_nodes))
    ].copy()

    catalog["name_clean"] = catalog["node_name"].fillna("").map(_clean)
    catalog = (catalog[catalog["name_clean"] != ""]
               .drop_duplicates("node_index_within_type", keep="first"))

    rows = []
    for _, row in catalog.iterrows():
        n    = row["name_clean"]
        ntok = set(n.split())
        ex   = float(n == q_norm)
        co   = float(q_norm in n or n in q_norm)
        ra   = difflib.SequenceMatcher(None, q_norm, n).ratio()
        ja   = len(q_tokens & ntok) / max(len(q_tokens | ntok), 1)
        rows.append({
            "disease_idx":  int(row["node_index_within_type"]),
            "disease_name": str(row["node_name"]),
            "match_score":  5.0 * ex + 2.5 * co + 2.0 * ra + 1.5 * ja,
        })

    return (pd.DataFrame(rows)
              .sort_values("match_score", ascending=False)
              .drop_duplicates("disease_idx")
              .reset_index(drop=True)
              .head(top_n))

match_df = match_disease(DISEASE_QUERY)
assert len(match_df) > 0, f"No disease found for '{DISEASE_QUERY}'."

selected_disease_idx  = int(match_df.iloc[0]["disease_idx"])
selected_disease_name = str(match_df.iloc[0]["disease_name"])

print(f"\nQuery   : '{DISEASE_QUERY}'")
print(f"Matched : {selected_disease_name}  (idx={selected_disease_idx})")
print("\nTop match candidates:")
display(match_df.head(5)[["disease_idx", "disease_name", "match_score"]])


# 3. Known treatment resolver

def get_known_treatments(drug_idx: int,
                         max_diseases: int = MAX_KNOWN_DISEASES) -> str:
    """
    Return a comma-separated string of diseases the drug already treats
    in the training graph.

    Why this matters for interpretation:
      If a drug treats coronary artery disease and we're recommending it
      for hypertension, the shared cardiovascular biology is immediately
      visible — it's a strong real-world repurposing signal.

    If the drug has no known training indications, it's flagged as
    'novel (no known indication)' — potentially a de-novo repurposing hit.
    """
    known = sorted(TRAIN_DISEASES_BY_DRUG.get(int(drug_idx), set()))
    if not known:
        return "novel (no known indication)"

    names = []
    for dis_idx in known[:max_diseases]:
        raw = _NODE_NAME_LOOKUP.get(
            (SUP_DST_TYPE.lower(), int(dis_idx)),
            f"disease_{dis_idx}"
        )
        names.append(str(raw))

    suffix = (f" (+{len(known) - max_diseases} more)"
              if len(known) > max_diseases else "")
    return ", ".join(names) + suffix

# 4. Raw drug retrieval

raw_df = score_all_drugs_for_disease(
    selected_model, graph, selected_disease_idx
).copy()

if EXCLUDE_TRAIN:
    known = TRAIN_THERAPEUTIC_BY_DISEASE.get(selected_disease_idx, set())
    raw_df = raw_df[~raw_df["drug_idx"].isin(known)]

if MIN_SCORE is not None:
    raw_df = raw_df[raw_df["score"] >= MIN_SCORE]

raw_df = raw_df.head(RAW_POOL).reset_index(drop=True)
print(f"\nRaw candidate pool : {len(raw_df)} drugs")


# 5. Evidence + Safety scoring helpers

def _evidence_strength(bundle: dict) -> dict:
    """
    Weighted sum of evidence counts → normalised strength score (0–1).
    Shared protein bridges are weighted highest because they represent
    a direct molecular mechanism; effect overlaps are weakest because
    they are phenotypic correlations only.
    """
    c   = bundle.get("evidence_counts", {})
    raw = (W_SHARED_PROT * c.get("shared_protein_bridge", 0) +
           W_PPI         * c.get("ppi_bridge", 0) +
           W_DD_TRANSFER * c.get("disease_disease_transfer", 0) +
           W_EFFECT      * c.get("effect_or_phenotype_overlap", 0))
    return {
        "evidence_strength_raw": float(raw),
        "evidence_strength":     float(min(raw / 10.0, 1.0)),
        **{f"{k}_count": int(v) for k, v in c.items()},
    }

def _safety_penalty(bundle: dict) -> dict:
    """
    Penalty score (0–1) from safety signals.
    Direct contraindications are penalised most heavily — a known
    contraindication edge in the graph is a hard signal.
    Zero-evidence drugs are also penalised lightly because there is
    no biological rationale to support the recommendation.
    """
    rows = bundle.get("safety_rows", [])
    n_c  = sum(1 for r in rows if r["safety_type"] == "direct_contraindication")
    n_a  = sum(1 for r in rows if r["safety_type"] == "adverse_effect_overlap")
    n_z  = 1 if bundle["n_evidence_rows"] == 0 else 0
    raw  = P_CONTRA * n_c + P_ADVERSE * n_a + P_ZERO_EV * n_z
    return {
        "safety_penalty_raw": float(raw),
        "safety_penalty":     float(min(raw / 8.0, 1.0)),
        "n_contra": n_c, "n_adverse": n_a, "zero_evidence": n_z,
    }


# 6. Reranking loop

rerank_rows    = []
rerank_bundles = []

print(f"\nBuilding evidence bundles for {len(raw_df)} candidates ...")

for _, row in raw_df.iterrows():
    drug_idx  = int(row["drug_idx"])
    raw_score = float(row["score"])

    bundle = extract_evidence_bundle(
        drug_idx, selected_disease_idx,
        model=selected_model, top_n_each=EVIDENCE_TOP_N,
    )
    ev = _evidence_strength(bundle)
    sf = _safety_penalty(bundle)
    fs = (W_MODEL * raw_score +
          W_EVIDENCE * ev["evidence_strength"] -
          W_SAFETY   * sf["safety_penalty"])

    rerank_rows.append({
        "drug_idx":        drug_idx,
        "drug_name":       bundle["drug_name"],
        "raw_model_score": raw_score,
        "rerank_score":    float(fs),
        "risk_level":      bundle["risk_level"],
        "n_evidence":      bundle["n_evidence_rows"],
        "n_safety":        bundle["n_safety_flags"],
        **ev, **sf,
    })
    rerank_bundles.append(bundle)

reranked_df = (pd.DataFrame(rerank_rows)
                 .sort_values(["rerank_score", "evidence_strength_raw",
                                "raw_model_score"], ascending=False)
                 .reset_index(drop=True))

# Build top-K with known_treatment_for
final_topk_df   = reranked_df.head(FINAL_TOP_K).copy()
_bundle_by_drug = {int(b["drug_idx"]): b for b in rerank_bundles}

final_topk_df["known_treatment_for"] = final_topk_df["drug_idx"].map(
    get_known_treatments
)

# Build detailed result objects
final_detailed_results = []
for rank, (_, rr) in enumerate(final_topk_df.iterrows(), start=1):
    drug_idx = int(rr["drug_idx"])
    bundle   = _bundle_by_drug[drug_idx]
    ev_df, sf_df = evidence_bundle_to_tables(bundle)
    final_detailed_results.append({
        "rank":               rank,
        "drug_idx":           drug_idx,
        "drug_name":          bundle["drug_name"],
        "known_treatment_for": str(rr["known_treatment_for"]),
        "raw_model_score":    float(rr["raw_model_score"]),
        "rerank_score":       float(rr["rerank_score"]),
        "risk_level":         bundle["risk_level"],
        "n_evidence":         bundle["n_evidence_rows"],
        "n_safety":           bundle["n_safety_flags"],
        "bundle":             bundle,
        "evidence_df":        ev_df,
        "safety_df":          sf_df,
        "rag_json":           bundle_to_rag_json(bundle),
    })

# Display
print(f"\n{'='*65}")
print(f"  TOP {FINAL_TOP_K} RERANKED DRUG CANDIDATES")
print(f"  Disease : {selected_disease_name}  |  Model : {selected_model_name}")
print(f"{'='*65}")
display(final_topk_df[[
    "drug_name",
    "raw_model_score", "rerank_score", "risk_level",
    "n_evidence", "n_safety",
    "known_treatment_for"
]].rename(columns={
    "drug_name":           "Drug",
    "raw_model_score":     "Model Score",
    "rerank_score":        "Rerank Score",
    "risk_level":          "Risk",
    "n_evidence":          "Evidence",
    "n_safety":            "Safety Flags",
    "known_treatment_for": "Known Treatment For",
}).reset_index(drop=True))


#Summary
print(f"\n{'='*55}")
print(f"  BLOCK 11 COMPLETE")
print(f"{'='*55}")
print("  Objects now available:")
print("    final_detailed_results  (includes known_treatment_for)")
print("    final_topk_df")
print("    reranked_df")
print("    match_disease(query)")
print("    get_known_treatments(drug_idx)")
print("    selected_disease_idx / selected_disease_name")
print("    selected_model / selected_model_name")
print(f"{'='*55}")

Using model : HGT

Query   : 'hypertension'
Matched : hypertension  (idx=1569)

Top match candidates:


,disease_idx,disease_name,match_score
0,1569,hypertension,11.000000
1,9289,renal hypertension,4.850000
2,14393,ocular hypertension,4.798387
3,4194,benign hypertension,4.798387
4,7122,pulmonary hypertension,4.661765



Raw candidate pool : 100 drugs

Building evidence bundles for 100 candidates ...

  TOP 5 RERANKED DRUG CANDIDATES
  Disease : hypertension  |  Model : HGT


,Drug,Model Score,Rerank Score,Risk,Evidence,Safety Flags,Known Treatment For
0,Aripiprazole (drug_1200),0.554106,0.660169,low,4,0,"bipolar disorder, schizophrenia, autism spectrum disorder (+1 more)"
1,Brigatinib (drug_6098),0.554003,0.600102,low,3,0,"non-small cell lung carcinoma (disease), lung cancer"
2,Pibrentasvir (drug_6424),0.554282,0.525283,low,2,0,"hepatitis C virus infection, chronic hepatitis C virus infection"
3,Mirodenafil (drug_5991),0.554073,0.525148,low,2,0,"psychologic dyspareunia, premature ejaculation (disease)"
4,Fluvoxamine (drug_162),0.554057,0.525137,low,2,0,"social phobia, phobic disorder, obsessive-compulsive disorder"



  BLOCK 11 COMPLETE
  Objects now available:
    final_detailed_results  (includes known_treatment_for)
    final_topk_df
    reranked_df
    match_disease(query)
    get_known_treatments(drug_idx)
    selected_disease_idx / selected_disease_name
    selected_model / selected_model_name


## 12. Disease similarity retrieval and disease-aware drug reranking

This block extends the reranking pipeline by adding **disease similarity** as another source of support for candidate drugs. This section adds one more idea: a drug may be a stronger candidate for the query disease if it already treats **other diseases that are biologically similar** to the query.

The final reranking idea used here is:

$$
\boxed{
\begin{aligned}
\mathbf{\text{final_score}} =\, &W_{\text{MODEL}} \times \text{raw_model_score} \\
+\, &W_{\text{EVIDENCE}} \times \text{evidence_strength} \\
+\, &W_{\text{SIMILAR}} \times \text{similar_disease_support} \\
-\, &W_{\text{SAFETY}} \times \text{safety_penalty}
\end{aligned}
}
$$

### 1. User settings and guards

The block starts by checking that the required outputs from earlier sections already exist. In particular, it depends on:

- the evidence extraction functions from **Block 10**
- the selected disease and selected model from **Block 11**
- the detailed candidate results already generated for the query disease

This section also defines the key user-controlled weights.

#### Similarity signal weights

Disease similarity is not computed from just one source. Instead, it combines six different signals:

- **embedding cosine similarity**
- **direct disease–disease graph links**
- **shared disease-associated proteins**
- **PPI-expanded protein overlap**
- **shared phenotype or effect overlap**
- **shared known therapeutic drugs**

Each of these signals gets its own weight, such as `W_SIM_EMBED`, `W_SIM_DD`, `W_SIM_PROT`, and so on. This prevents any single information source from dominating the similarity calculation.

#### Reranking weights

The block also sets weights for the final reranking formula:

- `W_MODEL`
- `W_EVIDENCE`
- `W_SIMILAR`
- `W_SAFETY`

These control how much influence each component has in the final disease-aware recommendation score.

Additional settings such as `TOP_K_SIMILAR`, `RAW_POOL`, `FINAL_TOP_K`, and `EVIDENCE_TOP_N` determine:
- how many similar diseases to retrieve
- how many raw drug candidates to consider
- how many final drugs to keep
- how much evidence detail to preserve

### 2. Build the disease embedding matrix

The first major step is to obtain a matrix of disease embeddings.

The helper function `_get_disease_embeddings()` returns a matrix of shape:

- **[number of diseases, embedding dimension]**

It first tries to use cached embeddings from the selected model, which is faster. If those are not available, it falls back to running the model encoder again to generate the embeddings directly.

This is important because one of the six disease-similarity signals is based on **cosine similarity in embedding space**. That signal reflects what the trained graph model has learned about disease relationships from the full heterogeneous graph structure.

### 3. Build biological neighborhood sets

The next step creates reusable biological feature sets for each disease. This is handled through helper functions such as `_build_sets(...)` and `_ppi_expand(...)`.

For a given disease, the code collects several kinds of neighborhood information:

- proteins directly associated with the disease
- proteins expanded by one hop through the protein–protein interaction network
- phenotypes or effects linked to the disease
- known therapeutic drugs for the disease
- directly linked neighboring diseases through disease–disease relations

These sets are used to compare the query disease against every other disease in the graph.

### 4. Compute the disease similarity table

This is the core disease retrieval part of the block.

For the selected query disease, the code compares it against every other disease and computes a weighted similarity score using six independent signals:

#### Embedding cosine similarity

This captures similarity in the learned representation space from the trained model. If two diseases occupy nearby positions in embedding space, they receive a higher similarity contribution.

#### Direct disease–disease graph edge

If the query disease and another disease are directly connected through disease hierarchy or ontology edges, they receive additional support.

#### Shared disease-associated proteins

This measures overlap in the proteins associated with the two diseases. A larger overlap suggests related molecular mechanisms.

#### PPI-expanded protein overlap

This is a softer version of protein overlap. Even if the diseases do not share the exact same proteins, they may still connect to nearby proteins in the protein–protein interaction network.

#### Shared phenotype or effect overlap

This checks whether the diseases are linked to similar phenotypes or effects, which provides another form of biological similarity.

#### Shared therapeutic drugs

This checks whether the diseases already share known treatment drugs. If they do, that can be a useful clue that the diseases may respond to overlapping interventions.

The code combines these six components into one final similarity score and stores the top results in:

- `query_similar_diseases_df`

This dataframe is the ranked table of diseases most similar to the selected query disease.

### 5. Compute similar-disease support for drugs

Once the similar diseases are identified, the block asks a second question:

**Does a candidate drug already treat any of these similar diseases?**

If the answer is yes, the drug gets a support boost.

This step works by checking each candidate drug against the retrieved similar diseases and summing the similarity scores of the diseases it already treats. So a drug receives stronger support when:

- it treats multiple similar diseases
- the treated diseases are very similar to the query disease
- the similarity evidence is strong across multiple signals

This produces a new measure called **similar disease support**, which acts as an additional evidence channel beyond the raw model score and graph evidence bundle.

### 6. Reuse evidence and safety helpers

Before final reranking, the block reuses the same evidence and safety logic introduced earlier. It defines helper functions to keep the scoring behavior aligned with Block 11.

These helpers still compute:

- **evidence strength**, based on supportive graph patterns such as shared proteins, PPI bridges, related-disease support, and phenotype overlap
- **safety penalty**, based on direct contraindications, adverse effect overlap, or lack of supporting evidence

### 7. Perform disease-aware drug reranking

This is the final step of the block.

The code starts from a raw pool of candidate drugs for the selected disease, usually obtained from the trained model’s scores. For each candidate drug, it then computes:

- the raw model score
- the graph evidence score
- the similar disease support score
- the safety penalty

These are combined using the weighted formula defined at the top of the block to produce a final reranked score.

The candidates are then sorted by this new final score, and the top results are stored in:

- `query_drug_recommendations_df`

A more detailed record is also kept in:

- `query_detailed_results`

This detailed output contains the per-drug breakdown of scores, evidence, and risk information so later blocks can display explanations alongside the ranked recommendations.

In [ ]:
# Disease Similarity Retrieval

# Guards
assert "extract_evidence_bundle"      in globals(), "Run Block 10 first."
assert "score_all_drugs_for_disease"  in globals(), "Run Block 10 first."
assert "TRAIN_THERAPEUTIC_BY_DISEASE" in globals(), "Run Block 10 first."
assert "selected_disease_idx"         in globals(), "Run Block 11 first."
assert "selected_disease_name"        in globals(), "Run Block 11 first."
assert "selected_model"               in globals(), "Run Block 11 first."
assert "final_detailed_results"       in globals(), "Run Block 11 first."

#User settings

# Similarity signal weights (sum ≤ 1)
W_SIM_EMBED  = 0.35   # embedding cosine similarity
W_SIM_DD     = 0.20   # direct disease-disease graph edge
W_SIM_PROT   = 0.20   # shared disease-associated proteins
W_SIM_PPI    = 0.10   # PPI-expanded protein bridge
W_SIM_EFFECT = 0.10   # shared phenotype / effect overlap
W_SIM_DRUG   = 0.05   # shared known treatment drugs

TOP_K_SIMILAR = 10    # similar diseases to retrieve

# Drug reranking weights
W_MODEL    = 0.55
W_EVIDENCE = 0.20
W_SIMILAR  = 0.25
W_SAFETY   = 0.15

RAW_POOL       = 100
FINAL_TOP_K    = 5
EVIDENCE_TOP_N = 5
EXCLUDE_TRAIN  = True
MIN_SCORE      = None

# Evidence / safety weights
W_SHARED_PROT = 3.0; W_PPI_W = 2.5; W_DD_W = 2.0; W_EFF_W = 1.0
P_CONTRA = 4.0; P_ADVERSE = 1.5; P_ZERO_EV = 1.5


# 1. Disease embedding matrix

@torch.no_grad()
def _get_disease_embeddings() -> np.ndarray:
    """
    Return the disease embedding matrix [num_diseases, D].
    Uses the cached {model}_embeddings dict if available (fast),
    falls back to a fresh model.encode() call if not.
    """
    key   = f"{'hgt' if selected_model_name == 'HGT' else 'baseline'}_embeddings"
    cache = globals().get(key)
    if isinstance(cache, dict) and SUP_DST_TYPE in cache:
        emb = cache[SUP_DST_TYPE]
        return emb.cpu().numpy() if torch.is_tensor(emb) else np.asarray(emb)
    z = selected_model.encode(
        graph.to(next(selected_model.parameters()).device)
    )
    return z[SUP_DST_TYPE].detach().cpu().numpy()

disease_emb_mat = _get_disease_embeddings()
query_emb       = disease_emb_mat[selected_disease_idx]
print(f"Disease embedding matrix : {disease_emb_mat.shape}")


# 2. Biological neighbour sets

def _build_sets(anchor_type: str, edge_types: list) -> dict:
    """
    For every node of anchor_type, collect all adjacent nodes
    reached via any of edge_types.
    Returns {anchor_idx: set of (neighbor_type, neighbor_idx)}.
    """
    sets: dict = defaultdict(set)
    for et in edge_types:
        s_t, _, d_t = et
        ei = graph[et].edge_index.detach().cpu()
        if anchor_type == s_t:
            for s, d in ei.t().tolist():
                sets[int(s)].add((d_t, int(d)))
        if anchor_type == d_t:
            for s, d in ei.t().tolist():
                sets[int(d)].add((s_t, int(s)))
    return sets

# Pre-build neighbour sets for all four signal types
DIS_PROTEINS = _build_sets(SUP_DST_TYPE, disease_protein_edge_types)
DIS_EFFECTS  = _build_sets(SUP_DST_TYPE, disease_effect_edge_types)
DIS_DD_NBRS  = _build_sets(SUP_DST_TYPE, disease_disease_edge_types)

# PPI bidirectional neighbour lookup
_PPI_NBRS: dict = defaultdict(set)
for _et in ppi_edge_types:
    _s_t, _, _d_t = _et
    _ei = graph[_et].edge_index.detach().cpu()
    for _s, _d in _ei.t().tolist():
        _PPI_NBRS[(_s_t, int(_s))].add((_d_t, int(_d)))
        _PPI_NBRS[(_d_t, int(_d))].add((_s_t, int(_s)))

def _ppi_expand(protein_set: set) -> set:
    """Expand a protein set by one PPI interaction hop."""
    expanded = set(protein_set)
    for p in protein_set:
        expanded |= _PPI_NBRS.get(p, set())
    return expanded

# Query disease fingerprint
_q_prots   = DIS_PROTEINS.get(selected_disease_idx, set())
_q_effects = DIS_EFFECTS.get(selected_disease_idx, set())
_q_dd      = DIS_DD_NBRS.get(selected_disease_idx, set())
_q_drugs   = TRAIN_THERAPEUTIC_BY_DISEASE.get(selected_disease_idx, set())
_q_prots_x = _ppi_expand(_q_prots)


# 3. Disease similarity table

def _log_norm(arr: np.ndarray) -> np.ndarray:
    """Log-normalise a non-negative array to [0, 1]."""
    max_v = float(arr.max())
    return np.log1p(arr) / np.log1p(max_v) if max_v > 0 else np.zeros_like(arr)

print(f"Computing similarity across "
      f"{int(graph[SUP_DST_TYPE].num_nodes):,} diseases ...")

rows = []
for cand in range(int(graph[SUP_DST_TYPE].num_nodes)):
    if cand == selected_disease_idx:
        continue

    c_prots   = DIS_PROTEINS.get(cand, set())
    c_effects = DIS_EFFECTS.get(cand, set())
    c_dd      = DIS_DD_NBRS.get(cand, set())
    c_drugs   = TRAIN_THERAPEUTIC_BY_DISEASE.get(cand, set())

    emb_cos   = float(
        np.dot(query_emb, disease_emb_mat[cand]) /
        max(np.linalg.norm(query_emb) * np.linalg.norm(disease_emb_mat[cand]),
            1e-12)
    )
    direct_dd = int(
        (SUP_DST_TYPE, cand) in _q_dd or
        (SUP_DST_TYPE, selected_disease_idx) in c_dd
    )
    n_prot    = len(_q_prots & c_prots)
    n_ppi     = len((_q_prots_x & c_prots) - _q_prots)
    n_effect  = len(_q_effects & c_effects)
    n_drug    = len(_q_drugs & c_drugs)

    rows.append({
        "similar_disease_idx":           int(cand),
        "similar_disease_name":          get_node_name(SUP_DST_TYPE, cand),
        "embedding_cosine":              emb_cos,
        "direct_disease_disease_edge":   direct_dd,
        "shared_protein_count":          n_prot,
        "ppi_bridge_count":              n_ppi,
        "shared_effect_count":           n_effect,
        "train_drug_overlap_count":      n_drug,
        "similar_known_train_drug_count": len(c_drugs),
        "sample_similar_train_drugs": ", ".join(
            _NODE_NAME_LOOKUP.get((SUP_SRC_TYPE.lower(), i), f"drug_{i}")
            for i in list(sorted(c_drugs))[:3]
        ),
    })

sim_df = pd.DataFrame(rows)

# Normalise signals and compute combined similarity score
sim_df["emb_norm"]    = (sim_df["embedding_cosine"].clip(-1, 1) + 1.0) / 2.0
sim_df["prot_norm"]   = _log_norm(sim_df["shared_protein_count"].values)
sim_df["ppi_norm"]    = _log_norm(sim_df["ppi_bridge_count"].values)
sim_df["effect_norm"] = _log_norm(sim_df["shared_effect_count"].values)
sim_df["drug_norm"]   = _log_norm(sim_df["train_drug_overlap_count"].values)

sim_df["combined_similarity_score"] = (
    W_SIM_EMBED  * sim_df["emb_norm"] +
    W_SIM_DD     * sim_df["direct_disease_disease_edge"] +
    W_SIM_PROT   * sim_df["prot_norm"] +
    W_SIM_PPI    * sim_df["ppi_norm"] +
    W_SIM_EFFECT * sim_df["effect_norm"] +
    W_SIM_DRUG   * sim_df["drug_norm"]
)

query_similar_diseases_df = (
    sim_df.sort_values(
        ["combined_similarity_score", "embedding_cosine",
         "shared_protein_count", "train_drug_overlap_count"],
        ascending=False
    ).head(TOP_K_SIMILAR).reset_index(drop=True)
)

print(f"\n{'='*65}")
print(f"  TOP {TOP_K_SIMILAR} MOST SIMILAR DISEASES")
print(f"  Query : {selected_disease_name}")
print(f"{'='*65}")
display(query_similar_diseases_df[[
    "similar_disease_name",
    "combined_similarity_score", "embedding_cosine",
    "shared_protein_count", "shared_effect_count",
    "train_drug_overlap_count"
]].rename(columns={
    "similar_disease_name":        "Similar Disease",
    "combined_similarity_score":   "Similarity Score",
    "embedding_cosine":            "Embedding Cos",
    "shared_protein_count":        "Shared Proteins",
    "shared_effect_count":         "Shared Effects",
    "train_drug_overlap_count":    "Shared Drugs",
}).reset_index(drop=True))


# 4. Similar-disease drug support

_drug_support_raw: dict = defaultdict(float)
_drug_support_dis: dict = defaultdict(list)

for _, row in query_similar_diseases_df.head(20).iterrows():
    sim_idx   = int(row["similar_disease_idx"])
    sim_score = float(row["combined_similarity_score"])
    sim_name  = str(row["similar_disease_name"])
    for drug_idx in TRAIN_THERAPEUTIC_BY_DISEASE.get(sim_idx, set()):
        _drug_support_raw[drug_idx]      += sim_score
        if len(_drug_support_dis[drug_idx]) < 5:
            _drug_support_dis[drug_idx].append(sim_name)

_max_support = max(_drug_support_raw.values()) if _drug_support_raw else 1.0

def _get_support(drug_idx: int):
    """Return (raw_support, normalised_support, supporting_disease_names)."""
    raw  = float(_drug_support_raw.get(drug_idx, 0.0))
    norm = raw / _max_support if _max_support > 0 else 0.0
    diss = " | ".join(_drug_support_dis.get(drug_idx, []))
    return raw, norm, diss


# 5. Evidence + Safety helpers

def _ev_strength(bundle):
    c   = bundle.get("evidence_counts", {})
    raw = (W_SHARED_PROT * c.get("shared_protein_bridge", 0) +
           W_PPI_W       * c.get("ppi_bridge", 0) +
           W_DD_W        * c.get("disease_disease_transfer", 0) +
           W_EFF_W       * c.get("effect_or_phenotype_overlap", 0))
    return {"evidence_strength_raw": float(raw),
            "evidence_strength":     float(min(raw / 10.0, 1.0))}

def _sf_penalty(bundle):
    rows = bundle.get("safety_rows", [])
    n_c  = sum(1 for r in rows if r["safety_type"] == "direct_contraindication")
    n_a  = sum(1 for r in rows if r["safety_type"] == "adverse_effect_overlap")
    n_z  = 1 if bundle["n_evidence_rows"] == 0 else 0
    raw  = P_CONTRA * n_c + P_ADVERSE * n_a + P_ZERO_EV * n_z
    return {"safety_penalty_raw": float(raw),
            "safety_penalty":     float(min(raw / 8.0, 1.0))}

# 6. Disease-aware drug reranking

raw_df = score_all_drugs_for_disease(
    selected_model, graph, selected_disease_idx
).copy()

if EXCLUDE_TRAIN:
    known = TRAIN_THERAPEUTIC_BY_DISEASE.get(selected_disease_idx, set())
    raw_df = raw_df[~raw_df["drug_idx"].isin(known)]
if MIN_SCORE is not None:
    raw_df = raw_df[raw_df["score"] >= MIN_SCORE]
raw_df = raw_df.head(RAW_POOL).reset_index(drop=True)

print(f"\nReranking {len(raw_df)} candidates with disease-similarity support ...")

rerank_rows    = []
rerank_bundles = []

for _, row in raw_df.iterrows():
    drug_idx  = int(row["drug_idx"])
    raw_score = float(row["score"])

    bundle = extract_evidence_bundle(
        drug_idx, selected_disease_idx,
        model=selected_model, top_n_each=EVIDENCE_TOP_N,
    )
    ev                 = _ev_strength(bundle)
    sf                 = _sf_penalty(bundle)
    s_raw, s_norm, s_d = _get_support(drug_idx)

    final = (W_MODEL    * raw_score +
             W_EVIDENCE * ev["evidence_strength"] +
             W_SIMILAR  * s_norm -
             W_SAFETY   * sf["safety_penalty"])

    rerank_rows.append({
        "drug_idx":                     drug_idx,
        "drug_name":                    bundle["drug_name"],
        "known_treatment_for":          get_known_treatments(drug_idx),
        "raw_model_score":              raw_score,
        "similar_disease_support":      s_norm,
        "similar_disease_support_raw":  s_raw,
        "supporting_similar_diseases":  s_d,
        "rerank_score":                 float(final),
        "risk_level":                   bundle["risk_level"],
        "n_evidence":                   bundle["n_evidence_rows"],
        "n_safety":                     bundle["n_safety_flags"],
        **ev, **sf,
    })
    rerank_bundles.append(bundle)

_reranked = (pd.DataFrame(rerank_rows)
               .sort_values(
                   ["rerank_score", "similar_disease_support_raw",
                    "evidence_strength_raw", "raw_model_score"],
                   ascending=False)
               .reset_index(drop=True))

query_drug_recommendations_df = _reranked.head(FINAL_TOP_K).copy()
_bundle_by_drug = {int(b["drug_idx"]): b for b in rerank_bundles}

# Build detailed result objects
query_detailed_results = []
for rank, (_, rr) in enumerate(query_drug_recommendations_df.iterrows(), start=1):
    drug_idx = int(rr["drug_idx"])
    bundle   = _bundle_by_drug[drug_idx]
    ev_df, sf_df = evidence_bundle_to_tables(bundle)
    query_detailed_results.append({
        "rank":                    rank,
        "drug_idx":                drug_idx,
        "drug_name":               bundle["drug_name"],
        "known_treatment_for":     str(rr["known_treatment_for"]),
        "raw_model_score":         float(rr["raw_model_score"]),
        "similar_disease_support": float(rr["similar_disease_support"]),
        "supporting_similar_diseases": str(rr["supporting_similar_diseases"]),
        "rerank_score":            float(rr["rerank_score"]),
        "risk_level":              bundle["risk_level"],
        "n_evidence":              bundle["n_evidence_rows"],
        "n_safety":                bundle["n_safety_flags"],
        "bundle":                  bundle,
        "evidence_df":             ev_df,
        "safety_df":               sf_df,
        "rag_json":                bundle_to_rag_json(bundle),
    })

print(f"\n{'='*65}")
print(f"  TOP {FINAL_TOP_K} DISEASE-AWARE DRUG RECOMMENDATIONS")
print(f"  Disease : {selected_disease_name}  |  Model : {selected_model_name}")
print(f"{'='*65}")
display(query_drug_recommendations_df[[
    "drug_name",
    "raw_model_score", "similar_disease_support",
    "rerank_score", "risk_level", "n_evidence",
    "known_treatment_for"
]].rename(columns={
    "drug_name":               "Drug",
    "raw_model_score":         "Model Score",
    "similar_disease_support": "Sim. Support",
    "rerank_score":            "Rerank Score",
    "risk_level":              "Risk",
    "n_evidence":              "Evidence",
    "known_treatment_for":     "Known Treatment For",
}).reset_index(drop=True))


# Summary
print(f"\n{'='*55}")
print(f"  BLOCK 12 COMPLETE")
print(f"{'='*55}")
print("  Objects now available:")
print("    query_similar_diseases_df")
print("    query_drug_recommendations_df  (with known_treatment_for)")
print("    query_detailed_results")
print("    _build_sets / _ppi_expand / _log_norm  (shared by Block 13)")
print(f"{'='*55}")

Disease embedding matrix : (17080, 128)
Computing similarity across 17,080 diseases ...

  TOP 10 MOST SIMILAR DISEASES
  Query : hypertension


,Similar Disease,Similarity Score,Embedding Cos,Shared Proteins,Shared Effects,Shared Drugs
0,hypertensive disorder (disease_12675),0.799431,0.996750,12,0,51
1,"essential hypertension, genetic (disease_15162)",0.733706,0.827450,12,0,2
2,schizophrenia (disease_12715),0.596113,0.975533,6,0,0
3,congestive heart failure (disease_12648),0.593885,0.997253,6,0,8
4,heart failure (disease_12852),0.579487,0.994419,6,0,2
5,anxiety disorder (disease_15492),0.577918,0.935944,6,0,0
6,kidney disease (disease_12844),0.571029,0.596279,1,0,0
7,unipolar depression (disease_12863),0.570691,0.908018,6,0,0
8,bipolar disorder (disease_12626),0.562907,0.867104,6,0,0
9,dysthymic disorder (disease_2746),0.554288,0.944373,5,0,0



Reranking 100 candidates with disease-similarity support ...

  TOP 5 DISEASE-AWARE DRUG RECOMMENDATIONS
  Disease : hypertension  |  Model : HGT


,Drug,Model Score,Sim. Support,Rerank Score,Risk,Evidence,Known Treatment For
0,Aripiprazole (drug_1200),0.554106,0.587499,0.651633,low,4,"bipolar disorder, schizophrenia, autism spectrum disorder (+1 more)"
1,Spironolactone (drug_403),0.553825,0.706262,0.521170,low,1,"Bartter disease, primary aldosteronism, congestive heart failure (+1 more)"
2,Risperidone (drug_707),0.554088,0.587499,0.501623,low,1,"autism susceptibility 1, manic bipolar affective disorder, bipolar disorder (+2 more)"
3,Amoxapine (drug_521),0.554136,0.573907,0.498252,low,1,"dysthymic disorder, neurotic disorder, anxiety disorder"
4,Desipramine (drug_1115),0.553715,0.573907,0.498020,low,1,"dysthymic disorder, neurotic disorder, anxiety disorder"



  BLOCK 12 COMPLETE
  Objects now available:
    query_similar_diseases_df
    query_drug_recommendations_df  (with known_treatment_for)
    query_detailed_results
    _build_sets / _ppi_expand / _log_norm  (shared by Block 13)
